# Public analysis copy

This notebook documents the analysis workflow from the original thesis notebook. Raw proteomic measurements, patient-level metadata, and private intermediate datasets are intentionally excluded from this public repository.

**Important:** the public notebook is not expected to run from a clean checkout without the restricted input data. Approved aggregate results and figures may be added separately to `results/`.


# Install packages 

In [ ]:
import os
import math
import random
import argparse
import re
import itertools
import warnings


import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib as mpl

from scipy.stats import (
    ttest_ind,
    kruskal,
    mannwhitneyu
)
from scipy.spatial.distance import pdist, squareform

from sklearn import decomposition
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs


import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

from skbio.stats.distance import permanova, DistanceMatrix

from kneed import KneeLocator

from adjustText import adjust_text

# Quality Control 

In [ ]:
# Function 1
def get_datamatrix(fragpipe_output): # fragpipe_output should be dir what output will be
    """retreive output from fragpipe, do some basic cleaning/formatting """ ## Triple Quotes for Docstring: Docstrings are string literals that appear as the first statement in a function, class, or module. These are used to explain what the code does and are enclosed in triple quotes
    try:
        pg= os.path.join(os.path.relpath(fragpipe_output), "report.pg_matrix.tsv")
        print(f"Retreiving from the following MS Fragger output for QC: {fragpipe_output}")
        pg = pd.read_csv(pg, sep="\t") # added to avoid the str error for using set_index on string and not a df
        pg.set_index('Protein.Group', inplace=True)
        ## fix column names to fit pnumber names
        samp_list= pg.columns.to_list() # drop(['Protein.Ids', 'Genes', 'First.Protein.Description'],
        samplist= [x.replace(' ', '').split("\\")[-1].rstrip(".d") for x in samp_list ] ## may want to further refine if the key sampID is in between underscores
        pg.columns= samplist
        ## fix some possible column name problems
        ## key error try / except?
        extra_cols= ['Protein.Ids' , 'Genes' , 'First.Protein.Description'] ## will this work if these cols not there?
        all_samps= [x.replace('__', '_').replace(' ','').split('_') for x in set(samplist) - set(extra_cols)] #if x.startswith(r'[0-9]')] ## remove spaces

        samp_map= pd.DataFrame(all_samps)

        samp_map.columns= ["plate_num", 'position', 'pnum', 'personal_plate_num', 'replicate', 'consec_num', "x", "consec_num_2"]
        samp_map['ms_consec_num'] = samp_map['consec_num_2'].fillna(samp_map['consec_num']) ## due to formatting issues
        ## correct og_name (original name) with ms consec number thing
        samp_map['og_name'] = samp_map["plate_num"].astype(str) + "_" + samp_map['position'].astype(str) + "_" + samp_map['pnum'] + "_" + samp_map['personal_plate_num']
        samp_map['og_name'] = samp_map['og_name'].replace(to_replace= "_None", value= ""
        , regex= True)
        samp_map.drop(columns= ['consec_num', 'x', 'consec_num_2'], axis= 1, inplace= True)

        return pg, samp_map

    except FileNotFoundError as e:
        print(f"Unable to detect the expected file: {e}")
        some_cat= get_cat(cat_type="sad")
        print(some_cat + "\n")
        cleanup_all_processes()
        sys.exit(1)

In [ ]:
# Function 2
def remove_contaminants(pg, reference):
    """ Remove common contaminants from proteomics experiments"""
    ## this source from the hao group seems quite reliable with diff sample type contaminants
    ## https://github.com/HaoGroup-ProtContLib/Protein-Contaminant-Libraries-for-DDA-and-DIA-Proteomics
    ## be sure to cites
    contam_dat= pd.read_csv("${THESIS_PRIVATE_DATA}/HaoGroup_ContaminantUniprotIDs_DDA_DIA_proteomics.tsv", sep="\t", skiprows= 1) ## in the same utils/ folder
    contam_dat.drop(['Status'], axis= 1, inplace= True)
    ## narrow down contam to that containing human data
    ## because i tested, otherwise we ID bovine contam in our data, which is relevant for cell cultures . .
    if "human" in reference:
        human_contam= contam_dat.loc[contam_dat['Organism'].str.contains("Homo sapiens", na = False)]
        contam_ids= list(human_contam['Uniprot ID'])
    elif "mouse" in reference:
        mouse_contam= contam_dat.loc[contam_dat['Organism'].str.contains("Mus musculus", na = False)]
        contam_ids= list(mouse_contam['Uniprot ID'])
    else:
        ## better to filter by host to avoid false pos hits, but otherwise, just use the entire list
        contam_ids= list(contam_dat['Uniprot ID'])
    ## pull out contams measured in our data
    measured_contaminants= pg.loc[pg['Protein.Ids'].isin(contam_ids)].copy()
    ## remove columns (samples) with no measured NAs.
    measured_contaminants.dropna(axis= 1, how= 'all', inplace= True)
    measured_contaminants= contam_dat.merge(measured_contaminants, how= 'right',
    right_on= 'Protein.Ids', left_on= 'Uniprot ID')
    ## write this to a seperate file in the results so ppl can ID on their own if they want to.
    contam_file_name= os.path.join(f"contaminants_ided_with_intensities_{reference}_source.csv") ## no outdir for nextflow file handling
    print("Private contaminant intensities are not exported in the public copy.")
    ## give some info to log file
    count_of_contams= len(measured_contaminants['Uniprot ID'].unique())
    print(f"Identified {count_of_contams} contaminants in the data;\nWriting IDed contaminants and intensities to files . . . ")
    ## now remove these contaminants from our data
    pg_filt= pg[~pg['Protein.Ids'].isin(contam_ids)]
    og_rows= pg.shape[0]
    new_rows= pg_filt.shape[0]
    ids_removed= og_rows- new_rows
    print(f"Applying contaminant filtering . . . ")#\n\tactual number of rows (IDs) removed is {ids_removed} (compared to the number of contaminants we IDs, {count_of_
    if ids_removed== count_of_contams:
        print(f"{count_of_contams} i.e. expected number of contaminants removed!")
    else:
        print(f"\tSomething weird happened; we identified {count_of_contams} contaminants, but removed {ids_removed} rows. Please manually inspect.")
    ## return filtered pg df
    return pg_filt

In [ ]:
# Function 3
def quantify_missingness(pg, missingness_threshold):
    """ quantify missingness in data """
    ## do we want to replace zero with the lowest value observed in the df near zero? (to prevent issues at next step)
    intensity_cols= pg.filter(regex="[0-9]", axis= 1).dropna(how= "all") ## axis - 1 for cols ; drop to remove peptides IDed but not quantified
    ## also, zeroes are valid values here too
    min_intensity= min(i for i in intensity_cols.stack().sort_values().head(n=5) if i > 0)
    print("Minimum detected intensity greater than zero: ", min_intensity) ## needs for whole DF!!
    # remove rows in which the PEPTIDE is observed in fewer than 70% of samples
    
    try:
        pg_filled= pg.dropna(thresh=(float(missingness_threshold)*(pg.filter(regex="[0-9]", axis= 1).shape[-1]))).copy() # thresh = 70% of samples if sampler cols start wit
        pg_filled.drop(['Genes', 'First.Protein.Description','Protein.Ids' ], axis= 1, inplace=True) # for pr: 'Proteotypic', 'Stripped.Sequence', 'Modified.Sequence
        new_name_pg_filled= os.path.join(f"report.pg_missingness_filled_at_{missingness_threshold}.csv") ## no outdir for netflow file handling
        print("Private proteomic matrix export disabled in the public copy.")

    
    except Exception as e:
        print(f"User provided missingness threshold is not detected as a number: {missingness_threshold}\nPlease edit the user parameter toml and try again.")
        some_cat= get_cat(cat_type="small")
        print(some_cat + "\n")
        cleanup_all_processes()
        sys.exit(1)
    
    pg_filled = pg_filled.fillna(min_intensity)
    return pg_filled

In [ ]:
# Function 4
def apply_log2(df, samp_map): ## samp_map is second output from get_datamatrix
    """ Applying log2 transformation and initial correction """
    ## corrected median intensity
    # corrected intensity is (original intensity / sample median) * mean of protein in batch (plate)
    df_t= df.transpose() ## want samples as rows, pg groups as columns

    df_t= df_t.drop(['Protein.Names']) #, 'Protein.Group', "First.Protein.Description", "Genes"
    df_t['name_from_intensity'] = df_t.index
    df_t['name_from_intensity'] = df_t.apply(lambda x: re.sub(r'[\W_]+','_', str(x['name_from_intensity'])), axis= 1)

    df_t['pnum'] = df_t.apply(lambda x: x['name_from_intensity'].split("_")[2], axis= 1)
    ## add the plate info


    df_t= pd.merge(df_t, samp_map, on = "pnum", how= "left") ## this adds plate num
    df_t_int= df_t.filter(regex="[0-9]", axis= 1).copy()

    metadata_cols = ['plate_num', 'position', 'pnum', 'personal_plate_num', 'replicate', 'ms_consec_num', 'og_name', 'name_from_intensity']
    protein_cols = [c for c in df_t.columns if c not in metadata_cols]
    df_t_int = df_t[protein_cols].copy()

    df_t_int.rename(columns=lambda s: s.replace(";", "_or_"), inplace=True)
    pg_names= df_t_int.columns.to_list()
    df_t_int.index= df_t['plate_num']
    ## cout no plate num . . . .
    df_t_int.loc[df_t_int.index.isna() =="True"]
    ## first, get the median of each sample within a batch, and ID the mean of medians
    ## mean of medians, but spread to two lines
    df_t_int.loc[:, 'row_median'] = df_t_int.apply(lambda x: x.median(), axis=1) ## sample median, calc without low values
    df_t_int.loc[:, 'group_mean'] = df_t_int.groupby('plate_num')['row_median'].transform('mean') ## batch mean of medians
    df_t_int.index= df_t['name_from_intensity'] ## to ensure downstream merging is to specific samples
    #df_t_int.head() ## samples as rows
    ### implementing a check: check out samples that aren't catched by the plate number coding
    werid_samples= df_t.loc[df_t.plate_num.isna()]
    count_in_both, count_not_in_exp= 0,0
    in_meta_dat_pnum= []
    for pnum in samp_map['pnum']:
        if pnum in werid_samples['pnum']:
            count_in_both += 1
        elif pnum not in werid_samples['pnum']:
            count_not_in_exp += 1
            in_meta_dat_pnum.append(pnum)

    ## note: some samples weird weird bc they were missing the QC label
    ## i'm keeping this check because it is a good sanity check, but there should be no more weird values
    print("The following number are in both meta data and our weird samples list: ", count_in_both)
    print("The following number are pnumbers in metadata but not in our weird samples list (aka normal behavior): ", count_not_in_exp)
    print("The following number are the pnumbers that are flagged by our weird sample list and not in the meta-data file: ", len(werid_samples['pnum'].unique()))
    print("This could be due to QC samples not being properly labelled, or something else with how you named your samples. The QC process will continue, but you may be missing samples . . . \n It is your responsibility to check.")
    #######################################################
    ## apply log2 transf; WITHIN-PLATE INTENSITY CORRECTION
    medians= df_t_int['row_median'] ## samples as rows, aka sample medians
    group_means= df_t_int['group_mean'] ## mean of plate medians
    # drop the last two columns for the intensity calculations
    df_t_int2= df_t_int[pg_names]
    corrected_intensities= (df_t_int2.div(medians, axis=0)).mul(group_means, axis=0)
    ## make columns numeric not object
    cols= corrected_intensities.columns
    corrected_intensities[cols] = corrected_intensities[cols].apply(pd.to_numeric, errors='coerce')
    ## drop na coolumns (unlikely but possible)
    corrected_intensities= corrected_intensities.dropna(axis= 1, how= "all")
    ##log2
    log2_pg= np.log2(corrected_intensities)
    log2_pg= log2_pg.round(6) ## samples as rows
    return log2_pg

In [ ]:
# Fucntion 5
def apply_batch_correction(log2_df):
    """ Returns dataframe with batch correction applied if the plate number is a part of the name of the samples."""
    ## ACROSS PLATES INTENSITY CORRECTION
    ## axis 0 in median for calc on columns aka protein groups
    log2_df.sort_index(inplace= True)
    log2_df['name_from_intensity'] = log2_df.index
    log2_df['plate_num'] = log2_df.apply(lambda x: x['name_from_intensity'].split("_")[0], axis= 1)
    log2_df.drop(['name_from_intensity'],axis= 1).groupby(['plate_num'])
    ## now calculate one median value per plate
    plate_medians= pd.DataFrame(log2_df.drop(['name_from_intensity'], axis= 1).groupby(['plate_num']).median()).apply(lambda x: x.median(),axis= 1)
    plates_mean= round(plate_medians.mean(),6)
    plate_medians= pd.DataFrame(plate_medians)
    plate_medians.columns= ['plate_median']
    ## probs faster way to make list as long as the OG df before the groupby>>median but here is whats in my brain rn
    log2_sub= log2_df.loc[:,['plate_num','name_from_intensity']]
    log2= pd.merge(log2_sub, plate_medians, on = "plate_num", how= "outer")
    log2.index= log2['name_from_intensity']
    log2.sort_index(inplace= True)
    ## now divide samples by plate median multuple by exp mean
    log2_batch_corr= log2_df.drop(['name_from_intensity','plate_num'], axis= 1).div(log2['plate_median'], axis= 0).mul(plates_mean, axis= 0)
    return log2_batch_corr

In [ ]:
# Function 6
def check_each_transformation(col_name, df1, df2, df3):
    ''' visualize difference that log2 transformation makes, only when batch correction is enabled'''
   

    fig, axs = plt.subplots(ncols=3)
    before_transf= sns.histplot(data= df1, x = col_name, kde= True, ax = axs[0]).set(title= "Before Transformation")
    after_transf= sns.histplot(data= df2.T, x = col_name, kde= True, ax = axs[1]).set(title= "After log2 Transf.")
    centered_transf= sns.histplot(data=df3.T, x = col_name, kde= True, ax = axs[2]).set(title= "After Batch Corr. ")
    fig.suptitle(f'Random Sample: {col_name}', fontsize=14)

    ## save to file
    plt_name= f"check_transformations_random_sample_{col_name}.png"
    plt.savefig(plt_name)
    return plt.show() ## won't show in pipeline?

In [ ]:
# Function 6.5
def check_each_transformation_nobc(col_name, df1, df2):
    ''' visualize difference that only log2 transformation makes'''
   

    fig, axs = plt.subplots(ncols=2)
    before_transf= sns.histplot(data= df1, x = col_name, kde= True, ax = axs[0]).set(title= "Before log2 Transformation")
    after_transf= sns.histplot(data= df2.T, x = col_name, kde= True, ax = axs[1]).set(title= "After log2 Transformation")
    fig.suptitle(f'Random Sample: {col_name}', fontsize=14)

    ## save to file
    plt_name= f"check_transformations_random_sample_{col_name}.png"
    plt.savefig('Before-After_log')
    return plt.show() ## won't show in pipeline?

In [ ]:
# Function 7
def apply_ms_drift_plot2(dat, samp_map, file_str= ""):
    """ Plot MS Drift - mean intensity across plate(s)"""
    print("there are a total of ", len(samp_map['plate_num'].unique()), " unique plates.")
    palette_len= len(samp_map['plate_num'].unique())
    custom_palette= sns.color_palette("tab20b", palette_len)
    # visualize log2 trans. on batches
    dat = dat.transpose().copy()
    qc_check= dat.describe()[:3]
    qc_check= qc_check.select_dtypes(include=['number']).transpose()
    qc_check['name_from_intensity'] = qc_check.index
    #qc_check['pnum'] = qc_check.apply(lambda x: x['name_from_intensity'].split("_")[2], axis= 1)
    qc_check['pnum'] = qc_check['name_from_intensity'].str.extract(r'(P\d+)')[0]
    qc_check= pd.merge(qc_check, samp_map, on = "pnum", how= "outer")
    ## now plot QC check to examine if we see mean log2 transf. MS Drift by ms consec num
    some_name= str(file_str)
    ## plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ms_drift_plot= sns.scatterplot(ax=ax,data= qc_check, x = "ms_consec_num", y= "mean", hue= "plate_num", palette= custom_palette)
    #specfiy axis labels
    ms_drift_plot.set(xlabel='MS Consec. Number',
        ylabel='Mean Intensity',
        title=f'MS Drift: {some_name}')
    ## make legend pretty
    ms_drift_plot.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
    ms_drift_plot.set(xticklabels=[]) # remove the tick labels
    ms_drift_plot.tick_params(bottom=False) # remove the ticks
    ms_drift_plot.figure.savefig(f"qc_ms_drift_{some_name}.png",  bbox_inches='tight')

In [ ]:
#Function 8
def apply_pca(dat, samp_map, file_str= ""):

    df_pca_data = dat.copy()

    sample_metadata = pd.DataFrame(index=df_pca_data.index)
    sample_metadata["sample_id"] = sample_metadata.index
    sample_metadata["pnum"] = sample_metadata["sample_id"].str.extract(r"(P\d+)")[0]

    sample_metadata = sample_metadata.merge(
        samp_map[["pnum", "plate_num"]].drop_duplicates(),
        on="pnum",
        how="left"
    )

    sample_metadata.index = sample_metadata["sample_id"]

    unique_plates = sample_metadata['plate_num'].unique()
    print("there are a total of ", len(unique_plates), " unique plates.")
    palette_len = len(unique_plates)
    custom_palette = sns.color_palette("tab20b", palette_len)
   
    pca = decomposition.PCA(n_components=2)
    df_pca_data = df_pca_data.fillna(0)

    
    pcs = pca.fit_transform(df_pca_data)

    pc_df = pd.DataFrame(
        pcs,
        columns=["PC1", "PC2"],
        index=df_pca_data.index
    )

    # Add metadata directly to plotting dataframe
    pc_df["plate_num"] = sample_metadata.loc[
        pc_df.index, "plate_num"
    ].fillna("Unknown")


    
    plt.figure(figsize=(8, 6))

    sns.scatterplot(
        data=pc_df,
        x="PC1",
        y="PC2",
        hue="plate_num",
        palette = custom_palette,
        alpha=0.8
    )
    

    plt.axhline(0, ls="--", alpha=0.5, color="black")
    plt.axvline(0, ls="--", alpha=0.5, color="black")

    plt.xlabel(
        f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", 
        fontsize = 14
    )
    plt.ylabel(
        f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", 
        fontsize = 14
    )

    plt.title(f"PCA: {file_str}", 
              fontsize = 16
             )
    plt.legend(
        title="Plate",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.tight_layout()

    plt.savefig(
        f"qc_pca_{file_str}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


In [ ]:
# PRIVATE DATA INPUT
# The original notebook loaded the raw FragPipe proteomics matrix here.
# Raw proteomic data are intentionally NOT included in this public repository.
#
# To run the analysis locally, place the private data outside the repository
# (or in the ignored private_data/ directory) and set the paths below.
#
# Example:
# PRIVATE_PROTEOMICS_DIR = os.environ["THESIS_PRIVATE_PROTEOMICS_DIR"]
# pg, samp_map = get_datamatrix(PRIVATE_PROTEOMICS_DIR)
#
# This public copy stops before loading restricted data.
print("Raw proteomic input is private and is not included in this repository.")

In [ ]:
# Log2 transform
pg_log2 = apply_log2(pg_filled, samp_map)

# Batch correction
pg_batch_correction = apply_batch_correction(pg_log2)

pg_log2 = pg_log2.drop(['name_from_intensity', 'plate_num'], axis=1)

In [ ]:
pg
# 1. Unique proteins in the whole cohort
total_cohort_proteins = pg.dropna(how='all').shape[0]

# 2. Unique proteins per sample
proteins_per_sample = pg.notna().sum(axis=0)

# Average across all samples
avg_proteins_per_sample = proteins_per_sample.mean()

print(f"Total unique proteins in whole cohort: {total_cohort_proteins}")
print(
    f"Average unique proteins per sample:   {avg_proteins_per_sample:.2f}"
    f" (± {proteins_per_sample.std():.2f})"
)

In [ ]:
# QC plots ‒ MS Drift
apply_ms_drift_plot2(
    pg_filled.drop(columns=['Protein.Names'], errors='ignore').T,
    samp_map,
    "Before log transformation"
)

apply_ms_drift_plot2(
    pg_log2.drop(columns=['name_from_intensity', 'plate_num'], errors='ignore'),
    samp_map,
    "After log transformation"
)

apply_ms_drift_plot2(
    pg_batch_correction,
    samp_map,
    "After batch correction"
)

# QC plots ‒ PCA

apply_pca(
    pg_batch_correction,
    samp_map,
    "After intial batch correction"
)


apply_pca(
    pg_batch_correction,
    samp_map,
    "After log2 transformation"
)

# QC plots ‒ Visualise transformation 

random_samples = random.sample(list(pg_filled.columns),1)

for sample in random_samples:
    check_each_transformation(sample, pg_filled, pg_log2, pg_batch_correction)

random_samples = random.sample(list(pg_filled.columns),1)

for sample in random_samples:
    check_each_transformation_nobc(sample, pg_filled, pg_log2)


In [ ]:
# PRIVATE DERIVED PROTEOMICS INPUT
# The original notebook loaded a processed proteomics matrix from a local file.
# That matrix is not included in the public repository.
print("Processed proteomic matrix is private and is not included in this repository.")

In [ ]:

apply_pca(
    pevids_log2_dat_pca,
    samp_map,
    "Final PCA on VICTORIA cohort"
)

# Descriptive Statistics of the clinical data

In [ ]:
# PRIVATE CLINICAL METADATA INPUT
# The original notebook loaded restricted clinical metadata and a data dictionary.
# Neither file is included in this public repository.
print("Clinical metadata and the data dictionary are private and are not included.")

In [ ]:
# The original notebook selected clinical variables from restricted metadata.
# This public copy does not reproduce patient-level metadata.
print("Patient-level metadata selection is intentionally omitted from the public copy.")

### sex

In [ ]:
### figure out the dummy coding used for sex in metadata - at index 14

sex = dic_dat[dic_dat['Unnamed: 0'] == 'sex']
sex_row = sex['Unnamed: 1'].values[0]
variable = sex['Unnamed: 0'].values[0]
print(f'The english coding for variable {variable} is: {sex_row}')

In [ ]:
mapping = {1: 'Female', 2: 'Male'}
meta_sub['sex'] = meta_sub['sex'].map(mapping)
meta_sub

In [ ]:
### check no. of males vs females 
sns.countplot(data=meta_sub, x='sex' , hue='sex', legend= False ,palette='muted')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.title("No. of female vs male patients")
plt.show()

### rapa - physical activity

In [ ]:
#dummy coding to groups
mapping = {(1, 2): 'Underactive', (3, 4, 5): 'Lightly active', (6, 7): 'Highly active'}
flat_mapping = {value: label for keys, label in mapping.items() for value in keys}
meta_sub['rapa_bc'] = meta_sub['rapa_bc'].map(flat_mapping)
meta_sub['rapa_bc']

In [ ]:
mapping = {(1.0, 2.0): 'Underactive', (3, 4, 5): 'Lightly active', (6, 7): 'Highly active'}
flat_mapping = {value: label for keys, label in mapping.items() for value in keys}
meta_sub['rapa_reha'] = meta_sub['rapa_reha'].map(flat_mapping)
meta_sub['rapa_reha']

In [ ]:
meta_sub.groupby('rapa_bc').count()

In [ ]:
meta_sub.groupby('rapa_reha').count()

### crc history

In [ ]:
mapping = {0: 'No', 1: 'Yes', 2: 'Unknown'}
meta_sub['b14mutter'] = meta_sub['b14mutter'].map(mapping)
meta_sub['b14mutter']

In [ ]:
mapping = {0: 'No', 1: 'Yes', 2: 'Unknown'}
meta_sub['b14vater'] = meta_sub['b14vater'].map(mapping)
meta_sub['b14vater']

In [ ]:
meta_sub.groupby('b14mutter').count()

In [ ]:
meta_sub.groupby('b14vater').count()

### smoking history / smoking current

In [ ]:
mapping = {0: 'No', 1: 'Yes'}
meta_sub['k33rauje'] = meta_sub['k33rauje'].map(mapping)
meta_sub['k33rauje']

In [ ]:
mapping = {0: 'No', 1: 'Yes'}
meta_sub['k35rauderzt'] = meta_sub['k35rauderzt'].map(mapping)
meta_sub['k35rauderzt']

In [ ]:
meta_sub.groupby('k33rauje').count()

In [ ]:
meta_sub.groupby('k35rauderzt').count()

### alcohol

In [ ]:
mapping = {0: 'Low risk alcohol consumption', 1: 'High Risk alcohol consumption'}
meta_sub['alc_class_vd'] = meta_sub['alc_class_vd'].map(mapping)
meta_sub['alc_class_vd']

In [ ]:
mapping = {0: 'Low risk alcohol consumption', 1: 'High Risk alcohol consumption'}
meta_sub['alc_class_rh'] = meta_sub['alc_class_rh'].map(mapping)
meta_sub['alc_class_rh']

In [ ]:
meta_sub.groupby('alc_class_vd').count()

In [ ]:
meta_sub.groupby('alc_class_rh').count()

### age

In [ ]:
sns.regplot(data=full_meta, x='bmi', y='vit_d', scatter_kws={'alpha':0.3, 'color':'coral'})
plt.show()

sns.regplot(data=full_meta, x='bmi', y='vit_d', scatter_kws={'alpha':0.5, 'color':'coral'}, x_bins=10)
plt.show()

In [ ]:
### Age group 
print(meta_sub['age'].describe())
sns.countplot(data=meta_sub, x='age')
plt.xticks(rotation=90)
plt.show()

sns.histplot(data=meta_sub, x='age', bins=20) 
plt.show()

sns.scatterplot(data=meta_sub, x='age', y='vit_d') # result - no clear linear relationship between age and vit d levels
plt.xlabel('Age')
plt.ylabel('Vit D levels')
plt.title('Vit D vs Age')
plt.show()

### fatigue

In [ ]:
fatigue_cols = [
        'PFA',
        'EFA',
        'CFA'
]

# clinically important thresholds (TCIs) for each fatigue dimension
fatigue_tci = {
    'PFA': 43,   # Physical fatigue TCI
    'EFA': 28,   # Emotional fatigue TCI
    'CFA': 25    # Cognitive fatigue TCI
}

# subset only fatigue variables
fatigue_df = meta_sub[fatigue_cols].copy()

# Check the number of missing values in each fatigue variable
print("Missing values:")
print(fatigue_df.isna().sum())
print(fatigue_df)

# Identify patients for whom all three fatigue dimensions are missing
all_fatigue_missing = fatigue_df[fatigue_cols].isna().all(axis=1)

# Report how many patients have no fatigue information available
print(
    "Patients with all fatigue variables missing:",
    all_fatigue_missing.sum()
)

In [ ]:
# Calculating each fatigue dimension relative to its own TCI
# A value of 1 means the patient's score is exactly at the TCI
# A value >1 means the score is above the TCI
# A value <1 means the score is below the TCI
fatigue_normalised = pd.DataFrame(index=fatigue_df.index)

# Normalise physical fatigue by its TCI of 43
fatigue_normalised['PFA_norm'] = fatigue_df['PFA'] / fatigue_tci['PFA']

# Normalise emotional fatigue by its TCI of 28
fatigue_normalised['EFA_norm'] = fatigue_df['EFA'] / fatigue_tci['EFA']

# Normalise cognitive fatigue by its TCI of 25
fatigue_normalised['CFA_norm'] = fatigue_df['CFA'] / fatigue_tci['CFA']


# Mean of the three TCI-normalised fatigue dimensions
meta_sub['fatigue_index'] = fatigue_normalised.mean(axis=1, skipna=True)

# Set the fatigue index to NA if all three fatigue variables are missing
meta_sub.loc[all_fatigue_missing, 'fatigue_index'] = np.nan


# Determine whether each fatigue dimension reaches or exceeds its clinical-importance threshold
fatigue_above_tci = pd.DataFrame(index=fatigue_df.index)

# Classify physical fatigue as clinically important if PFA >= 43
fatigue_above_tci['PFA'] = fatigue_df['PFA'] >= fatigue_tci['PFA']

# Classify emotional fatigue as clinically important if EFA >= 28
fatigue_above_tci['EFA'] = fatigue_df['EFA'] >= fatigue_tci['EFA']

# Classify cognitive fatigue as clinically important if CFA >= 25
fatigue_above_tci['CFA'] = fatigue_df['CFA'] >= fatigue_tci['CFA']


# Count the number of fatigue dimensions reaching their respective TCI
# pandas sum() treats True as 1 and False as 0
# skipna=True prevents missing values from being counted as positive fatigue
meta_sub['fatigue_domains'] = fatigue_above_tci.sum(axis=1, skipna=True)

# Set the number of fatigue domains to missing when all three dimensions are missing
meta_sub.loc[all_fatigue_missing, 'fatigue_domains'] = np.nan


# Convert the number of clinically important fatigue domains into categories
# 0 = no fatigue domain reaches its TCI -> minimal
# 1 = one fatigue domain reaches its TCI -> moderate
# 2 = two or more fatigue domains reach their TCIs -> severe
meta_sub['fatigue_mapped'] = pd.cut(
    meta_sub['fatigue_domains'],
    bins=[-1, 0, 1, 3],
    labels=[
        'minimal',
        'moderate',
        'severe'
    ],
    include_lowest=True
)

meta_sub[
    fatigue_cols + [
        'fatigue_index',
        'fatigue_domains',
        'fatigue_mapped'
    ]
]

### social score

In [ ]:
social_cols = [
    "c15hilfa",
    "c15hilfb",
    "c15hilfc",
    "c15hilfd",
    "c15hilfe"
]

social_df = meta_sub[social_cols].copy()
social_df

print(social_df.isna().sum())
social_df

df1 = social_df[social_df.isna().any(axis=1)]
print(df1)

meta_sub['social_score'] = social_df.mean(axis=1)
meta_sub

In [ ]:
bins1 = [1, 2.33, 3.67 , 5]
labels1 = ['minimal', 'moderate', 'great']

meta_sub['social_mapped'] = pd.cut(
    meta_sub['social_score'], bins=bins1, labels=labels1, include_lowest=True
)

meta_sub['social_mapped'].replace(0, np.nan)


### healthy lifestyle score (dietary)

In [ ]:
mapping = {(0, 1): 'Unhealthy', (2, 3): 'Avg. Healthy', (4, 5): 'Healthy'}
flat_mapping = {value: label for keys, label in mapping.items() for value in keys}
meta_sub['healthy_score_vd'] = meta_sub['healthy_score_vd'].map(flat_mapping)
meta_sub['healthy_score_vd']

In [ ]:
mapping = {(0, 1): 'Unhealthy', (2, 3): 'Avg. Healthy', (4, 5): 'Healthy'}
flat_mapping = {value: label for keys, label in mapping.items() for value in keys}
meta_sub['healthy_score_reha'] = meta_sub['healthy_score_reha'].map(flat_mapping)
meta_sub['healthy_score_reha']

### fraility

In [ ]:
# frail score
mapping = {(0,): 'Robust', (1, 2): 'Pre fraility', (3, 4, 5): 'Frail'}
flat_mapping = {value: label for keys, label in mapping.items() for value in keys}
meta_sub['total_scorefrail'] = meta_sub['total_scorefrail'].map(flat_mapping)
meta_sub['total_scorefrail']

### therapy

In [ ]:
meta_sub["op"] = meta_sub["p63op"] == 1
meta_sub["chemo"] = meta_sub["p63chemo"] == 1
meta_sub["radio"] = meta_sub["p63radio"] == 1
meta_sub

In [ ]:
conditions = [
    meta_sub["op"] & ~meta_sub["chemo"] & ~meta_sub["radio"],
    ~meta_sub["op"] & meta_sub["chemo"] & ~meta_sub["radio"],
    ~meta_sub["op"] & ~meta_sub["chemo"] & meta_sub["radio"],
    meta_sub["op"] & meta_sub["chemo"] & ~meta_sub["radio"],
    meta_sub["op"] & ~meta_sub["chemo"] & meta_sub["radio"],
    ~meta_sub["op"] & meta_sub["chemo"] & meta_sub["radio"],
    meta_sub["op"] & meta_sub["chemo"] & meta_sub["radio"],
]

choices = [
    "operation",
    "chemotherapy",
    "radiotherapy",
    "operation+chemo",
    "operation+radio",
    "chemo+radio",
    "all_three"
]

meta_sub["therapy_type"] = np.select(conditions, choices, default="none")

### Saving files

In [ ]:
# Sensitive metadata export removed from the public repository.
print("Private metadata are not exported by the public notebook.")

### Plots

In [ ]:
def add_count_percent_labels(ax):
    total = sum(container.datavalues.sum() for container in ax.containers)

    for container in ax.containers:
        labels = [
            f"{int(v)} ({100*v/total:.1f}%)"
            for v in container.datavalues
        ]
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)

In [ ]:
# Public copy: descriptive statistics are not recomputed from private patient-level data.
# Final, approved aggregate results can be placed in results/tables/ if desired.
print("Aggregate descriptive results are kept separate from the public source data.")

In [ ]:

sns.set_theme(
    style="ticks",
    context="talk"
)

fig, ax = plt.subplots(figsize=(7,5))

sns.histplot(
    data=meta_sub,
    x="age",
    bins=15,
    kde=True,
    ax=ax
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("Patients")
ax.set_title("Age Distribution")

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(
    style="whitegrid",
    context="paper"
)

fig = plt.figure(figsize=(14,10))

gs = fig.add_gridspec(
    3,
    2,
    hspace=0.6,
    wspace=0.3
)

# -------------------
# BMI
# -------------------
ax1 = fig.add_subplot(gs[0,0])

sns.histplot(
    meta_sub["bmi"],
    bins=15,
    kde=True,
    ax=ax1
)

ax1.set_title("BMI")
ax1.text(-0.1, 1.1, 'A', transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

# -------------------
# Smoking
# -------------------
ax2 = fig.add_subplot(gs[0,1])

sns.countplot(
    data=meta_sub,
    x="k33rauje",
    ax=ax2
)

add_count_percent_labels(ax2)

ax2.set_title("Smoking History")
ax2.set_xlabel("")
ax2.text(-0.1, 1.1, 'B', transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

# -------------------
# Frailty
# -------------------
ax3 = fig.add_subplot(gs[1,0])

sns.countplot(
    data=meta_sub,
    x="total_scorefrail",
    order=["Robust","Pre fraility","Frail"],
    ax=ax3
)
add_count_percent_labels(ax3)

ax3.set_title("Frailty")
ax3.text(-0.1, 1.1, 'C', transform=ax3.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

# -------------------
# Fatigue
# -------------------
ax4 = fig.add_subplot(gs[1,1])

sns.countplot(
    data=meta_sub,
    x="fatigue_mapped",
    order=["minimal","moderate","severe"],
    ax=ax4
)
add_count_percent_labels(ax4)

ax4.set_title("Fatigue")
ax4.text(-0.1, 1.1, 'D', transform=ax4.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

# -------------------
# Physical Activity
# -------------------
ax5 = fig.add_subplot(gs[2,0])

sns.countplot(
    data=meta_sub,
    x="rapa_bc",
    order = ["Underactive","Lightly active","Highly active"],
    ax=ax5
)
add_count_percent_labels(ax5)

ax5.set_title("Physical Activity")
ax5.text(-0.1, 1.1, 'E', transform=ax5.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

# Social Score
ax6 = fig.add_subplot(gs[2,1])

sns.countplot(
    data=meta_sub,
    x="healthy_score_vd",
    ax=ax6
)
add_count_percent_labels(ax6)

ax6.set_title('Healthy Lifestyle Score')
ax6.text(-0.1, 1.1, 'F', transform=ax6.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')

fig.suptitle(
    "Baseline Clinical and Lifestyle Characteristics of CRC Cohort (n=301)",
    fontsize=18
)

for ax in fig.axes:
    # Get current y-axis maximum
    ymin, ymax = ax.get_ylim()
    # Scale top limit by 15% to make room for text labels
    ax.set_ylim(ymin, ymax * 1.15)

plt.tight_layout()
plt.show()

fig.savefig('ds_panel.png', dpi=300, bbox_inches='tight')

# Data Merge

In [ ]:
# Private curated metadata input removed from public copy.
print("Curated metadata are private and are not included in this repository.")

In [ ]:
protein_cols = pevids_log2_dat.loc[:, "A0A075B6H7":"Q9Y6R7"].columns
print(protein_cols)

In [ ]:
curated_meta_bl = curated_meta_dat[(curated_meta_dat['Messzeitpunkt'] == 1.0) & (curated_meta_dat['data_source'] == 'victoria')]
curated_meta_bl

# OR drop duplicates based on specific key columns
clean_curated_bl = curated_meta_bl.drop_duplicates(subset=['sample_id'], keep='first')

print(f"Original shape: {curated_meta_bl.shape}")
print(f"Cleaned shape: {clean_curated_bl.shape}")


In [ ]:
# This merge uses restricted proteomic and metadata inputs and is therefore not
# executed in the public copy.
print("Proteomic/metadata merge omitted from public execution.")

In [ ]:
# This merge creates a patient-level dataset and is intentionally not executed
# or stored in the public repository.
print("Patient-level merged dataset omitted from public execution.")

In [ ]:
imp_cols = merged_meta_dat.loc[:, "A0A075B6H7":].columns
merged_meta = merged_meta_dat[['P_number'] + list(imp_cols)]
merged_meta_dat = merged_meta.set_index('P_number')
merged_meta_dat # 301 rows or samples orginally 

In [ ]:
# Sensitive merged-data export removed from the public repository.
print("Private merged data are not exported by the public notebook.")

In [ ]:
protein_cols = merged_meta_dat.loc[:, "A0A075B6H7":"Q9Y6R7"].columns
print(protein_cols)

# Statistical Testing 

## Functions: PERMANOVA, Mann-Whitney U-Test and Volcano Plots, K-means clustering

In [ ]:
def apply_permanova(col_name):
    '''Applying PERMANOVA and pairwise testing using Euclidean distance'''

    # Removing NAs
    nona_meta_dat = merged_meta_dat[merged_meta_dat[col_name].notna()] 
    num_nona = len(nona_meta_dat.index)
    num_total = len(merged_meta_dat.index)
    print(f'Number of rows: {num_nona}')
    print(f'Number of NAs removed: {num_total - num_nona}')
    group = nona_meta_dat[col_name]

   
    # general PERMANOVA
    dist_dat = nona_meta_dat[list(protein_cols)]
    euclid_dist = squareform(pdist(dist_dat, metric="euclidean"))
    dist_mat = DistanceMatrix(euclid_dist, ids=nona_meta_dat.index.astype(str))
    result = permanova(dist_mat, nona_meta_dat[col_name])
    print(result)

    # Pairwise testing
    groups = nona_meta_dat[col_name].unique()
    results = []

    for g1, g2 in itertools.combinations(groups, 2):
    
        idx = nona_meta_dat[col_name].isin([g1, g2])
    
        sub_meta = nona_meta_dat.loc[idx]
        sub_dm = dist_mat.filter(sub_meta.index) 

        res = permanova(sub_dm, sub_meta, column= col_name, permutations=999)
    
        results.append({
            "group1": g1,
            "group2": g2,
            "pseudo-F": res['test statistic'],
            "p-value": res['p-value']
        })

    df = pd.DataFrame(results)
    results

    df["p-adjusted"] = multipletests(df["p-value"], method="fdr_bh")[1]
    print(df)

    return df, nona_meta_dat

In [ ]:
def apply_kmeans(meta_nona):
    kmeans = KMeans(init="random", n_clusters=3, n_init=10, max_iter=300, random_state=42)
    X = meta_nona[list(protein_cols)]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    sse = []
    
    for k in range(1, 11):
    
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
    
        kmeans.fit(X_scaled)
    
        sse.append(kmeans.inertia_)
    
    plt.plot(range(1, 11), sse, marker='o')
    
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("SSE")
    plt.title("Elbow Method")
    
    plt.show()
    
    for k in range(2, 11):
    
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
    
        labels = kmeans.fit_predict(X_scaled)
    
        score = silhouette_score(X_scaled, labels)
    
        print(f'k={k}, silhouette={score:.3f}')


In [ ]:
def apply_kmeans_plot(clus, meta_nona, col_name):
    """Plotting the clusters"""
    file_path = "thesis_figures/K-means plot"

    # Ensure output directory exists to avoid FileNotFoundError
    os.makedirs(file_path, exist_ok=True)

    kmeans = KMeans(n_clusters=clus, random_state=42, n_init=10)

    X = meta_nona[list(protein_cols)]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clusters = kmeans.fit_predict(X_scaled)

    # Use .loc to avoid SettingWithCopyWarning if meta_nona is a slice
    meta_nona = meta_nona.copy()
    meta_nona["kmeans_cluster"] = clusters
    print(meta_nona[["kmeans_cluster"]].head())

    ct = pd.crosstab(meta_nona[col_name], meta_nona["kmeans_cluster"])
    print(ct)
    print(meta_nona[["kmeans_cluster", col_name]])

    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)

    # FIX 1: Capture the figure object explicitly (fig, ax)
    fig, ax = plt.subplots(figsize=(8, 6))

    scatter = ax.scatter(pcs[:, 0], pcs[:, 1], c=clusters, cmap="Set1")

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"{col_name} - K-means clustering")

    # Save figure in both formats using the 'fig' handle
    for ext in ("png", "pdf"):
        save_kwargs = {"bbox_inches": "tight"}
        if ext == "png":
            save_kwargs["dpi"] = 600

        fig.savefig(
            os.path.join(file_path, f"{col_name}_k_means.{ext}"), **save_kwargs
        )

    plt.show()
    plt.close(fig)

In [ ]:
def apply_utest(df,
    protein_cols,
    group_col,
    group1,
    group2,):
    '''non-parametric Mann-Whitney U-Test'''


    fdr_thresh = 0.05
    

    results = []

    for protein in protein_cols:


        g1 = df.loc[
            df[group_col] == group1,
            protein
        ]

        g2 = df.loc[
            df[group_col] == group2,
            protein
        ]


        stat, pval = mannwhitneyu(
            g1,
            g2,
            alternative="two-sided"
        )

        log2fc = (np.median(g2) - np.median(g1)
        )

        results.append({
            "Protein": protein,
            "log2FC": log2fc,
            "pval": pval
        })


    
    results = pd.DataFrame(results)

    results["FDR"] = multipletests(
        results["pval"],
        method="fdr_bh"
    )[1]

    results["neglog10FDR"] = -np.log10(
        results["FDR"]
    )

    results["Significant"] = (
        (results["FDR"] < fdr_thresh)
    )

    return results

In [ ]:
#### theres more functions in final_v1 for volcano plot

In [ ]:
def plot_volcano(results, var, top_n_labels=20):


    file_path = "thesis_figures/Volcano Plots"
    required_cols = {"log2FC", "neglog10FDR", "Significant", "Protein"}
    missing = required_cols - set(results.columns)
    if missing:
        raise ValueError(f"results is missing required columns: {missing}")

    fdr_thresh = 0.05
    log2fc_thresh = 0.58

    mpl.rcParams["pdf.fonttype"] = 42       # keeps text as editable text in Illustrator
    mpl.rcParams["ps.fonttype"] = 42
    mpl.rcParams["svg.fonttype"] = "none"
    mpl.rcParams["axes.linewidth"] = 1.2

    df = results.copy()
    df["Dual_Sig"] = (df["Significant"] == True) & (df["log2FC"].abs() >= log2fc_thresh)

    # split into up / down / not-significant for clearer, conventional coloring
    conditions = [
        df["Dual_Sig"] & (df["log2FC"] > 0),
        df["Dual_Sig"] & (df["log2FC"] < 0),
    ]
    df["Direction"] = np.select(conditions, ["Up", "Down"], default="NS")

    palette = {"Up": "#d62728", "Down": "#1f77b4", "NS": "#b0b0b0"}
    plot_order = ["NS", "Down", "Up"]  # draw NS first so hits sit on top

    fig, ax = plt.subplots(figsize=(7, 6), dpi=300)

    for cat in plot_order:
        sub = df[df["Direction"] == cat]
        label = "Not significant" if cat == "NS" else cat
        ax.scatter(
            sub["log2FC"],
            sub["neglog10FDR"],
            s=18 if cat == "NS" else 28,
            c=palette[cat],
            alpha=0.5 if cat == "NS" else 0.85,
            linewidths=0,
            label=f"{label} (n={len(sub)})",
            zorder=2 if cat == "NS" else 3,
        )

    # threshold lines
    ax.axhline(-np.log10(fdr_thresh), linestyle="--", color="black", linewidth=0.8, zorder=1)
    ax.axvline(log2fc_thresh, linestyle="--", color="black", linewidth=0.8, zorder=1)
    ax.axvline(-log2fc_thresh, linestyle="--", color="black", linewidth=0.8, zorder=1)


    sig = df[df["Dual_Sig"]].sort_values("neglog10FDR", ascending=False).head(top_n_labels)

    texts = [
        ax.text(row["log2FC"], row["neglog10FDR"], row["Protein"], fontsize=8, fontstyle="italic")
        for _, row in sig.iterrows()
    ]
    if texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="grey", lw=0.5))

    ax.set_xlabel(r"log$_2$ fold change", fontsize=14)
    ax.set_ylabel(r"-log$_{10}$(FDR)", fontsize=14)
    ax.set_title(f"{var} — Volcano Plot (Mann–Whitney U test)", fontsize=15, pad=12)
    ax.tick_params(labelsize=11)

    # clean axes — no top/right box
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # symmetric x-axis, at least ±1
    xmin, xmax = ax.get_xlim()
    x_lim = max(abs(xmin), abs(xmax), 1)
    ax.set_xlim(-x_lim, x_lim)

    ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()


    for ext in ("png", "pdf"):
        fig.savefig(
            os.path.join(file_path, f"{var}_volcano.{ext}"),
            dpi=600 if ext == "png" else None,
            bbox_inches="tight",
        )

    plt.show()
    return fig, ax

## Apply functions

### rapa_bc

In [ ]:
rapa_bc_perm, rapa_bc_nona = apply_permanova('rapa_bc')

rapa_bc_binary = rapa_bc_nona[list(protein_cols) + ['rapa_bc']]
rapa_bc_binary = rapa_bc_binary[rapa_bc_binary['rapa_bc'].isin(['Underactive', 'Highly active'])]

rapa_bc_results = apply_utest(rapa_bc_binary, protein_cols, 'rapa_bc', 'Underactive', 'Highly active')
print(rapa_bc_results)

plot_volcano(rapa_bc_results, 'Rapid Assessment of Physical Acitivity ‒ Before Rehabilitation')

In [ ]:
apply_kmeans(rapa_bc_nona)
apply_kmeans_plot(5, rapa_bc_nona, 'rapa_bc')

### rapa_reha 

In [ ]:
rapa_reha_perm, rapa_reha_nona = apply_permanova('rapa_reha')

rapa_reha_binary = rapa_reha_nona[list(protein_cols) + ['rapa_reha']]
rapa_reha_binary = rapa_reha_binary[rapa_reha_binary['rapa_reha'].isin(['Underactive', 'Lightly active'])]

rapa_reha_results = apply_utest(rapa_reha_binary, protein_cols, 'rapa_reha', 'Underactive', 'Lightly active')
print(rapa_reha_results)

plot_volcano(rapa_reha_results, 'Rapid Assessment of Physical Activity: After Rehabilitation')

In [ ]:
print(rapa_reha_results[rapa_reha_results['Significant']])

### social_mapped 
(NS permanova)

In [ ]:
social_perm, social_nona = apply_permanova('social_mapped') # not significant 

### smoking now (k35rauderzt)
(NS permanova)

In [ ]:
k35rauderzt_perm, k35rauderzt_nona = apply_permanova('k35rauderzt')
smoking_now_binary = k35rauderzt_nona[list(protein_cols) + ['k35rauderzt']]
smoking_now_binary
smoking_now_results = apply_utest(smoking_now_binary, protein_cols, 'k35rauderzt', 'Yes', 'No')
print(smoking_now_results)
plot_volcano(smoking_now_results, 'Current Smoking Status')


In [ ]:
apply_kmeans(k35rauderzt_nona)
apply_kmeans_plot(2, k35rauderzt_nona, 'k35rauderzt')

In [ ]:
print(smoking_now_results[smoking_now_results['Significant']])

### smoking history (k33rauje)

In [ ]:
k33rauje_perm, k33rauje_nona = apply_permanova('k33rauje')
smoking_binary = k33rauje_nona[list(protein_cols) + ['k33rauje']]
smoking_results = apply_utest(smoking_binary, protein_cols, 'k33rauje', 'Yes', 'No')
print(smoking_results)
plot_volcano(smoking_results, 'Smoking History Status')
print(smoking_results[smoking_results['Significant']])

In [ ]:
apply_kmeans(k33rauje_nona)
apply_kmeans_plot(3, k33rauje_nona, 'k33rauje')

### fatigue 
(NS permanova)

In [ ]:
fatigue_perm, fatigue_nona = apply_permanova('fatigue_mapped')

### healthy score vd

In [ ]:
healthy_vd_perm, healthy_vd_nona = apply_permanova('healthy_score_vd')
healthy_binary = healthy_vd_nona[list(protein_cols) + ['healthy_score_vd']]
healthy_binary = healthy_binary[healthy_binary['healthy_score_vd'].isin(['Healthy', 'Unhealthy'])]
healthy_binary

healthy_results = apply_utest(healthy_binary, protein_cols, 'healthy_score_vd', 'Unhealthy', 'Healthy')
print(healthy_results)
plot_volcano(healthy_results, 'Healthy Score: Before Rehabilitation')

In [ ]:
print(healthy_results[healthy_results['Significant']])

In [ ]:
vd_plot = healthy_binary[['P00738' , 'P0DOX3', 'healthy_score_vd']]
vd_plot

In [ ]:
plt.figure(figsize=(6,4))

sns.histplot(data=vd_plot, x='P00738', hue='healthy_score_vd',
             kde=True, bins=10, element="step")

plt.title("Distribution of ProteinA expression")
plt.show()

In [ ]:
# Boxplot
import matplotlib.pyplot as plt
import seaborn as sns

p_adj = 0.032

plt.figure(figsize=(6, 5))

# 1. Boxplot (showfliers=False removes duplicate points)
ax = sns.boxplot(
    x="healthy_score_vd",
    y="P00738",
    data=vd_plot,
    showfliers=False,
    color="#2b7bba",
)

# 2. Stripplot 
sns.stripplot(
    x="healthy_score_vd",
    y="P00738",
    data=vd_plot,
    color="black",
    alpha=0.6,
    jitter=0.2,
)

y_min = vd_plot["P00738"].min()
y_max = vd_plot["P00738"].max()
y_range = y_max - y_min

plt.text(
    x=0.5,
    y=y_max + (y_range * 0.08),
    s=f"adj. p = {p_adj:.3f}",
    ha="center",
    va="bottom",
    fontsize=10,
)

plt.ylim(bottom=y_min - (y_range * 0.1), top=y_max + (y_range * 0.25))

# Formatting
plt.title("Haptoglobin expression across conditions", fontsize=14)
plt.xlabel("Groups", fontsize=12)
plt.ylabel("Protein abundance", fontsize=12)

plt.savefig(
    "Haptoglobin_expression_boxplot.png",
    dpi=600,
    bbox_inches="tight",
    transparent=False,
)

plt.tight_layout()
plt.show()

### healthy score reha

In [ ]:
healthy_reha_perm, healthy_reha_nona = apply_permanova('healthy_score_reha')
healthy_binary_reha = healthy_reha_nona[list(protein_cols) + ['healthy_score_reha']]
healthy_binary_reha = healthy_binary_reha[healthy_binary_reha['healthy_score_reha'].isin(['Healthy', 'Unhealthy'])]

healthy_results_reha = apply_utest(healthy_binary_reha, protein_cols, 'healthy_score_reha', 'Unhealthy', 'Healthy')
print(healthy_results_reha)
plot_volcano(healthy_results_reha, 'Healthy Score: After Rehabilitation')


In [ ]:
print(healthy_results_reha[healthy_results_reha['Significant']])

In [ ]:
reha_plot = healthy_binary_reha[['P02743' , 'Q15848', 'healthy_score_reha']]
reha_plot

In [ ]:
apply_kmeans(healthy_reha_nona)
apply_kmeans_plot(2, healthy_reha_nona, 'healthy_score_reha')

In [ ]:
p_adj = 0.006

plt.figure(figsize=(6, 5))

ax = sns.boxplot(
    x="healthy_score_reha",
    y="P02743",
    data=reha_plot,
    showfliers=False,
    color="#2b7bba",
)
sns.stripplot(
    x="healthy_score_reha",
    y="P02743",
    data=reha_plot,
    color="black",
    alpha=0.6,
    jitter=0.2,
)

# Position text relative to max value
y_min = reha_plot["P02743"].min()
y_max = reha_plot["P02743"].max()
y_range = y_max - y_min

# Add adj p-value text centered at x = 0.5
plt.text(
    x=0.5,
    y=y_max + (y_range * 0.1),
    s=f"adj. p = {p_adj:.3f}",
    ha="center",
    va="bottom",
    fontsize=10,
)

plt.ylim(bottom=y_min - (y_range * 0.1), top=y_max + (y_range * 0.25))

plt.title(
    "Serum amyloid P-component\nexpression across conditions", fontsize=14
)
plt.xlabel("Groups", fontsize=12)
plt.ylabel("Protein abundance", fontsize=12)

plt.savefig(
    "SAP_expression_boxplot.png",
    dpi=600,
    bbox_inches="tight",
    transparent=False,
)

plt.tight_layout()
plt.show()

In [ ]:
p_adj = 0.006

plt.figure(figsize=(6, 5))

# 1. Boxplot with showfliers=False (removes duplicate outlier points)
ax = sns.boxplot(
    x="healthy_score_reha",
    y="Q15848",
    data=reha_plot,
    showfliers=False,
    color="#2b7bba",
)
sns.stripplot(
    x="healthy_score_reha",
    y="Q15848",
    data=reha_plot,
    color="black",
    alpha=0.6,
    jitter=0.2,
)

y_min = reha_plot["Q15848"].min()
y_max = reha_plot["Q15848"].max()
y_range = y_max - y_min

plt.text(
    x=0.5,
    y=y_max + (y_range * 0.08),
    s=f"adj. p = {p_adj:.3f}",
    ha="center",
    va="bottom",
    fontsize=10,
)

plt.ylim(bottom=y_min - (y_range * 0.1), top=y_max + (y_range * 0.25))

# Formatting
plt.title("Adiponectin expression\nacross conditions", fontsize=14)
plt.xlabel("Groups", fontsize=12)
plt.ylabel("Protein abundance", fontsize=12)

plt.savefig(
    "Adinopectin_expression_boxplot.png",
    dpi=600,
    bbox_inches="tight",
    transparent=False,
)

plt.tight_layout()
plt.show()

### frail score

In [ ]:
total_scorefrail_perm, total_scorefrail_nona = apply_permanova('total_scorefrail')

In [ ]:
frail_binary = total_scorefrail_nona[list(protein_cols) + ['total_scorefrail']]
frail_binary = total_scorefrail_nona[total_scorefrail_nona['total_scorefrail'].isin(['Robust', 'Frail'])]
frail_binary

In [ ]:
frail_results = apply_utest(frail_binary, protein_cols, 'total_scorefrail', 'Robust', 'Frail')
print(frail_results)
plot_volcano(frail_results, 'Fraility')

In [ ]:
print(frail_results[frail_results['Significant']])

### therapy type

In [ ]:
therapy_perm, therapy_nona = apply_permanova('therapy_type')

In [ ]:
therapy_binary2 = therapy_nona[list(protein_cols) + ['therapy_type']]
therapy_binary2 = therapy_nona[therapy_nona['therapy_type'].isin(['all_three', 'operation+chemo'])]


In [ ]:
therapy_binary = therapy_nona[list(protein_cols) + ['therapy_type']]
therapy_binary = therapy_nona[therapy_nona['therapy_type'].isin(['operation', 'operation+chemo'])]
therapy_binary

t2_results = apply_utest(therapy_binary, protein_cols, 'therapy_type', 'operation', 'operation+chemo')
print(t2_results)
plot_volcano(t2_results, 'therapy')

### PCA (rapa_bc)

In [ ]:
end_idx = list(rapa_bc_nona.columns).index("Q9Y6R7")
target_columns = rapa_bc_nona.columns[: end_idx + 1]
print(target_columns)

In [ ]:
group_col = 'rapa_bc'
proteins = [c for c in target_columns if c != group_col]

groups = rapa_bc_nona[group_col].unique()

# KRUSKAL-WALLIS TEST PER PROTEIN

results = []

for protein in proteins:
    data_groups = [
        rapa_bc_nona[rapa_bc_nona[group_col] == g][protein].dropna()
        for g in groups
    ]
    

    stat, p = kruskal(*data_groups)

    results.append({
        "protein": protein,
        "H_stat": stat,
        "p_value": p
    })

res = pd.DataFrame(results)

# FDR CORRECTION

res["p_adj"] = multipletests(res["p_value"], method="fdr_bh")[1]

# SIGNIFICANT PROTEINS
sig = res[res["p_adj"] < 0.05].sort_values("p_adj")

print("Significant proteins:", sig.shape[0])
print(sig.head(20))

#### rapa_bc is not univariant 

In [ ]:

X = rapa_bc_nona[protein_cols]
groups = rapa_bc_nona.loc[X.index, 'rapa_bc']

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
pca_df['rapa_bc'] = groups.values

plt.figure(figsize=(8,6))

for g in pca_df["rapa_bc"].unique():
    subset = pca_df[pca_df["rapa_bc"] == g]
    plt.scatter(subset["PC1"], subset["PC2"], label=g, alpha=0.7)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("PCA: Protein expression structure by RAPA score")
plt.legend()
plt.tight_layout()
plt.show()

print("Top contributing proteins to PC1:")
loadings = pd.Series(pca.components_[0], index=X.columns)
print(loadings.abs().sort_values(ascending=False).head(20))

# Robust Linear Models (RLM)

In [ ]:
# MODEL 2
# Physical Fatigue ~ Physical Activity
# Outcome:
#   PFA (Physical Fatigue)

# Main Predictor:
#   scorephy_reha
#
# Confounders:
#   age           -> fatigue increases with age
#   sex           -> fatigue differs by sex
#   bmi           -> obesity influences fatigue
#   therapy_type  -> treatment burden may influence fatigue

rlm_pfa_activity = smf.rlm(
    "PFA ~ scorephy_reha + age + bmi + C(sex) + C(therapy_type)",
    data=meta
).fit()

print("\nMODEL 2: Physical Fatigue ~ Physical Activity")
print(rlm_pfa_activity.summary())


# MODEL 3
# Emotional Fatigue ~ Frailty
# Outcome:
#   EFA (Emotional Fatigue)
#
# Main Predictor:
#   total_scorefrail
#
# Confounders:
#   age
#   sex
#   bmi
#   therapy_type


rlm_efa = smf.rlm(
    "EFA ~ total_scorefrail + age + bmi + C(sex) + C(therapy_type)",
    data=meta
).fit()

print("\nMODEL 3: Emotional Fatigue ~ Frailty")
print(rlm_efa.summary())

# MODEL 4
# Cognitive Fatigue ~ Frailty

# Outcome:
#   CFA (Cognitive Fatigue)
#
# Main Predictor:
#   total_scorefrail
#
# Confounders:
#   age
#   sex
#   therapy_type

rlm_cfa = smf.rlm(
    "CFA ~ C(total_scorefrail) + age + C(sex) + C(therapy_type)",
    data=meta
).fit()

print("\nMODEL 4: Cognitive Fatigue ~ Frailty")
print(rlm_cfa.summary())



# MODEL 5
# Interference with Daily Life ~ Physical Fatigue

# Outcome:
#   IDL
#
# Main Predictor:
#   PFA
#
# Confounders:
#   age
#   sex
#   bmi


rlm_idl = smf.rlm(
    "IDL ~ PFA + age + bmi + C(sex)",
    data=meta
).fit()

print("\nMODEL 5: Daily Life Interference ~ Physical Fatigue")
print(rlm_idl.summary())



# MODEL 6
# Social Sequelae ~ Emotional Fatigue

# Outcome:
#   SOC
#
# Main Predictor:
#   EFA
#
# Confounders:
#   total_scorefrail
#   age
#   sex


rlm_soc = smf.rlm(
    "SOC ~ EFA + total_scorefrail + age + C(sex)",
    data=meta
).fit()

print("\nMODEL 6: Social Sequelae ~ Emotional Fatigue")
print(rlm_soc.summary())


# MODEL 8
# Physical Fatigue ~ Vitamin D
# Outcome:
#   PFA
#
# Main Predictor:
#   vit_d
#
# Confounders:
#   age
#   sex
#   bmi
#   therapy_type
#
# Biological hypothesis:
# Lower vitamin D may be associated with increased fatigue.

rlm_vitd_fatigue = smf.rlm(
    "PFA ~ vit_d + age + bmi + C(sex) + C(therapy_type)",
    data=meta
).fit()

print("\nMODEL 8: Physical Fatigue ~ Vitamin D")
print(rlm_vitd_fatigue.summary())




In [ ]:
# MODEL 1
# Vitamin D ~ Frailty
# Outcome:
#   vit_d
#
# Main Predictor:
#   total_scorefrail
#
# Confounders:
#   sex           -> sex differences in vitamin D metabolism
#   bmi           -> vitamin D is influenced by adiposity
#   therapy_type  -> treatment may affect nutrition/activity

rlm_vitd = smf.rlm(
    "vit_d ~ C(total_scorefrail) + bmi + C(sex) + C(therapy_type)",
    data=meta
).fit()

print("\nMODEL 1: Vitamin D ~ Frailty")
print(rlm_vitd.summary())

In [ ]:
# MODEL 7
# Frailty ~ Healthy Lifestyle Score
# Outcome:
#   total_scorefrail
#
# Main Predictor:
#   healthy_score_reha
#
# Confounders:
#   age
#   sex
#   bmi
#   therapy_type

rlm_frailty = smf.rlm(
    "Ctotal_scorefrail ~ C(healthy_score_reha) + age + bmi + C(sex) + C(therapy_type)",
    data=meta
).fit()

print("\nMODEL 7: Frailty ~ Healthy Lifestyle")
print(rlm_frailty.summary())

In [ ]:
# MODEL 9
# Physical Fatigue ~ Vitamin D

# Outcome:
#   PFA
#
# Main Predictor:
#   vit_d
#
# Confounders:
#   age
#   sex
#   bmi
#   therapy_type



frailty_map = {
    'Robust': 0,
    'Pre fraility': 1,
    'Frail': 2
}

meta['frailty_score'] = meta['total_scorefrail'].map(frailty_map)

# Robust Linear Model
rlm_frailty_fatigue = smf.rlm(
    "frailty_score ~ PFA + age + bmi + C(sex) + C(therapy_type)",
    data=meta
).fit()


print("\nMODEL 9: Fraility ~ Physcial Fatigue")
print(rlm_frailty_fatigue.summary())




In [ ]:
meta['total_scorefrail']

In [ ]:
# Private metadata input removed from public copy.
print("Additional private metadata loading omitted from the public copy.")

In [ ]:

rlm_frailty_fatigue = smf.rlm(
    "healthy_vd_num ~ fatigue_index + bmi + vit_d + social_score + C(sex) + C(therapy_type)",
    data=meta
).fit()


print("\nMODEL 10: Healthy Score (vd) ~ Fatigue")
print(rlm_frailty_fatigue.summary())



# Robust Linear Model
rlm_frailty_fatigue = smf.rlm(
    "healthy_reha_num ~ fatigue_index + bmi + vit_d + social_score + C(sex) + C(therapy_type)",
    data=meta
).fit()


print("\nMODEL 11: Healthy Score (reha) ~ Fatigue")
print(rlm_frailty_fatigue.summary())


